# Colab 3: Combined Two-Dataset Training

This notebook clones the repo, installs dependencies, finds two dataset folders by real CSV files, trains the combined model, evaluates it, zips the outputs, copies the zip to Drive, and downloads it.

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
REPO_URL = 'https://github.com/au510621104021/FND_2027.git'
BRANCH = 'main'
PROJECT_DIR = '/content/FND_2027'

# Search roots for datasets. Add more paths here if needed.
DATASET_SEARCH_ROOTS = [
    '/content/drive/MyDrive',
    '/content'
]

# Name hints used when selecting the two datasets.
DATASET_1_HINTS = ['isot fake news dataset', 'isot']
DATASET_2_HINTS = ['fake news dataset']

# Leave as None to auto-discover. If auto-discovery still picks the wrong folders,
# paste the exact paths here and rerun from this cell.
DATASET_DIR_1 = None
DATASET_DIR_2 = None

DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/FND_2027_outputs'

In [32]:
import os
from pathlib import Path

os.chdir('/content')
!rm -rf /content/FND_2027
!git clone --branch main https://github.com/au510621104021/FND_2027.git /content/FND_2027

assert Path(PROJECT_DIR).exists(), f'Project directory not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Project directory:', os.getcwd())
!ls

Cloning into '/content/FND_2027'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 224 (delta 53), reused 58 (delta 23), pack-reused 117 (from 1)
Receiving objects: 100% (224/224), 57.86 MiB | 14.54 MiB/s, done.
Resolving deltas: 100% (96/96), done.
Updating files: 100% (47/47), done.
Project directory: /content/FND_2027
 app			        Fake_News_Project_File_by_File_Explained.pptx
 colab2.ipynb		       'Fake or Real News Dataset'
 colab3.ipynb		       'ISOT Fake News Dataset'
 colab.ipynb		        README.md
 config			        requirements.txt
'Fake News Dataset'	        scripts
'Fake News Detection Dataset'   src


In [33]:
import os
from pathlib import Path

os.chdir(PROJECT_DIR)
assert Path('requirements.txt').exists(), 'requirements.txt not found. Clone step likely failed.'
!pip install -q -r requirements.txt

In [34]:
from pathlib import Path

CSV_NAMES = {'train.csv', 'test.csv', 'dataset.csv', 'data.csv', 'Fake.csv', 'True.csv'}

def folder_has_dataset_csvs(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    names = {p.name for p in path.glob('*.csv')}
    return bool(names.intersection(CSV_NAMES))

def collect_dataset_candidates(search_roots):
    candidates = []
    seen = set()
    for root_str in search_roots:
        root = Path(root_str)
        if not root.exists():
            continue
        for csv_file in root.rglob('*.csv'):
            parent = csv_file.parent.resolve()
            if str(parent) in seen:
                continue
            if folder_has_dataset_csvs(parent):
                seen.add(str(parent))
                candidates.append(parent)
    return sorted(candidates, key=lambda p: str(p).lower())

def pick_candidate(candidates, hints):
    scored = []
    for path in candidates:
        text = str(path).lower()
        score = 0
        for hint in hints:
            if hint in path.name.lower():
                score += 5
            if hint in text:
                score += 2
        if score > 0:
            scored.append((score, len(path.parts), str(path)))
    if not scored:
        return None
    scored.sort(key=lambda x: (-x[0], x[1], x[2]))
    return scored[0][2]

candidates = collect_dataset_candidates(DATASET_SEARCH_ROOTS)
print('Detected dataset candidate folders:')
for candidate in candidates:
    print(' -', candidate)

if DATASET_DIR_1 is None:
    DATASET_DIR_1 = pick_candidate(candidates, DATASET_1_HINTS)
if DATASET_DIR_2 is None:
    DATASET_DIR_2 = pick_candidate(candidates, DATASET_2_HINTS)

print('\nSelected DATASET_DIR_1 =', DATASET_DIR_1)
print('Selected DATASET_DIR_2 =', DATASET_DIR_2)

assert DATASET_DIR_1 is not None, 'Could not find dataset 1 automatically. Use one of the printed candidate folders.'
assert DATASET_DIR_2 is not None, 'Could not find dataset 2 automatically. Use one of the printed candidate folders.'
assert Path(DATASET_DIR_1).exists(), f'Folder not found: {DATASET_DIR_1}'
assert Path(DATASET_DIR_2).exists(), f'Folder not found: {DATASET_DIR_2}'

print('\nDataset 1 CSVs:', [p.name for p in Path(DATASET_DIR_1).glob('*.csv')])
print('Dataset 2 CSVs:', [p.name for p in Path(DATASET_DIR_2).glob('*.csv')])

Detected dataset candidate folders:
 - /content/FND_2027/Fake News Dataset
 - /content/FND_2027/Fake News Detection Dataset
 - /content/FND_2027/Fake or Real News Dataset
 - /content/FND_2027/ISOT Fake News Dataset

Selected DATASET_DIR_1 = /content/FND_2027/ISOT Fake News Dataset
Selected DATASET_DIR_2 = /content/FND_2027/Fake News Dataset

Dataset 1 CSVs: ['train.csv', 'test.csv']
Dataset 2 CSVs: ['train.csv', 'test.csv']


In [35]:
import os
import yaml
from pathlib import Path

os.chdir(PROJECT_DIR)
config_path = Path('config/config.yaml')
with config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['dataset_name'] = 'generic'
config['data']['data_dir'] = DATASET_DIR_1
config['data']['dataset_names'] = ['generic', 'generic']
config['data']['data_dirs'] = [DATASET_DIR_1, DATASET_DIR_2]
config['logging']['checkpoint_dir'] = './checkpoints'
config['logging']['log_dir'] = './logs'
config['inference']['model_checkpoint'] = './checkpoints/best_model_combined.pt'

with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Updated config for combined training:')
print(config['data'])

Updated config for combined training:
{'augment_images': True, 'data_dir': '/content/FND_2027/ISOT Fake News Dataset', 'dataset_name': 'generic', 'data_dirs': ['/content/FND_2027/ISOT Fake News Dataset', '/content/FND_2027/Fake News Dataset'], 'dataset_names': ['generic', 'generic'], 'image_augmentation': {'color_jitter': True, 'horizontal_flip': True, 'normalize': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]}, 'random_crop': True}, 'num_workers': 2, 'pin_memory': True}


In [36]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

CUDA available: True
GPU name: Tesla T4


In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/train.py --config config/config.yaml

2026-04-03 16:12:37.618448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775232757.641221   64230 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775232757.648408   64230 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775232757.666688   64230 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775232757.666712   64230 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775232757.666716   64230 computation_placer.cc:177] computation placer alr

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
best_model = Path('checkpoints/best_model.pt')
combined_model = Path('checkpoints/best_model_combined.pt')
assert best_model.exists(), 'Training did not produce checkpoints/best_model.pt'
shutil.copy2(best_model, combined_model)
print(f'Saved combined checkpoint to: {combined_model}')

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/evaluate.py --config config/config.yaml --checkpoint checkpoints/best_model_combined.pt

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
zip_base = Path('combined_model_artifacts')
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir='.', base_dir='checkpoints')
print(f'Created zip: {zip_path}')
!zip -ur combined_model_artifacts.zip results logs -x "*.ipynb_checkpoints*"

In [ ]:
import os
from pathlib import Path
from google.colab import files

os.chdir(PROJECT_DIR)
zip_path = Path('combined_model_artifacts.zip')
assert zip_path.exists(), 'Zip file was not created.'

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
!cp combined_model_artifacts.zip "$DRIVE_OUTPUT_DIR/"
print(f'Copied zip to Drive: {DRIVE_OUTPUT_DIR}')

files.download(str(zip_path))